# 🔁 Epoch, Batch Size & Iterations — Notes + Interview
---
> **Simple English** | **Interview Ready**

## 📌 Three Key Training Terms

### Epoch
- One **complete pass** through the entire training dataset
- Epoch = 1 cycle: the model sees every training sample once
- Training uses multiple epochs (10, 50, 100, 200...)
- More epochs → more learning (but risk of overfitting)

### Batch Size
- Number of training samples used **in one forward+backward pass**
- Common values: 8, 16, 32, 64, 128, 256
- Smaller batch → noisier but sometimes better generalization
- Larger batch → faster (GPU parallel), more stable

### Iterations (Steps)
- Number of **batches needed** to complete 1 epoch
- `Iterations = Total Samples ÷ Batch Size`

## 🔑 Formula
```
Iterations per Epoch = Total Training Samples / Batch Size

Example:
  1000 samples, batch_size = 32
  → Iterations = 1000 / 32 ≈ 32 iterations per epoch
  → With 50 epochs → 50 × 32 = 1600 total weight updates
```

## 📊 Batch Size Comparison
| Batch Size | Speed | Memory | Stability | Generalization |
|---|---|---|---|---|
| 1 (SGD) | Fast steps | Low | Noisy | Sometimes better |
| 32–64 | Balanced ✅ | Medium | Good | Good |
| Full dataset | Slow steps | High | Very stable | Sometimes worse |

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)

n_samples   = len(X_tr)  # 800
batch_size  = 32
epochs      = 10
iterations  = n_samples // batch_size  # per epoch

print("=== Training Configuration ===")
print(f"Training samples  : {n_samples}")
print(f"Batch size        : {batch_size}")
print(f"Epochs            : {epochs}")
print(f"Iterations/Epoch  : {n_samples}/{batch_size} = {iterations}")
print(f"Total weight updates: {epochs} × {iterations} = {epochs*iterations}")

In [ ]:
# Train and observe epoch-by-epoch progress
def build_model():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1,  activation='sigmoid')
    ])

model = build_model()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Training with batch_size=32, epochs=20:")
history = model.fit(X_tr, y_tr,
                    epochs=20,
                    batch_size=32,
                    validation_split=0.1,
                    verbose=1)
loss, acc = model.evaluate(X_te, y_te, verbose=0)
print(f"\n✅ Test Accuracy: {acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'],     label='Train Loss', color='blue')
axes[0].plot(history.history['val_loss'], label='Val Loss',   color='red')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='Train Acc', color='blue')
axes[1].plot(history.history['val_accuracy'], label='Val Acc',   color='red')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare different batch sizes
import matplotlib.pyplot as plt

batch_sizes = [8, 32, 256]
colors = ['blue', 'green', 'red']
plt.figure(figsize=(12, 4))

for bs, col in zip(batch_sizes, colors):
    m = build_model()
    m.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    h = m.fit(X_tr, y_tr, epochs=20, batch_size=bs, validation_split=0.1, verbose=0)
    plt.plot(h.history['val_accuracy'], label=f'batch={bs}', color=col)

plt.title('Validation Accuracy vs Batch Size')
plt.xlabel('Epoch'); plt.ylabel('Val Accuracy')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()
print("Small batch = noisy but sometimes reaches better accuracy")
print("Large batch = stable but can get stuck")

## 🗣️ Interview Questions & Answers

**Q: What is an epoch?**
> One complete pass through the entire training dataset. After each epoch the model has seen every sample once and weights have been updated many times.

**Q: What is the difference between batch size and iterations?**
> Batch size = samples per update. Iterations = updates per epoch = total_samples / batch_size. Example: 1000 samples, batch=32 → ~31 iterations per epoch.

**Q: Why is batch size 32 or 64 common?**
> Powers of 2 align well with GPU memory. These sizes are big enough for stable gradients but small enough for GPU to fit in memory. Good balance of speed and stability.

**Q: What happens if you use too many epochs?**
> Model starts memorizing training data — **overfitting**. Val loss starts increasing while train loss keeps decreasing. Solution: early stopping, dropout.